In [ ]:
"""
Example: Automatic config loading with ESN_model
"""
import numpy as np
from config.esn_config import auto_load_or_create, save_esn_model_to_config, find_matching_config, ESNConfig
from models_data_driven import ESN_model


# ========== Method 1: Using auto_load_or_create (Recommended) ==========
print("=" * 60)
print("Method 1: Using auto_load_or_create")
print("=" * 60)

t = np.linspace(0, 10, 1000)
data = np.sin(t)[np.newaxis, :, np.newaxis]

# First run: trains new model
print("\n>>> First call (will train):")
esn1 = auto_load_or_create(
    data=data,
    dt=0.01,
    N_units=20,
    config_dir="./saved_configs",
    auto_save=True,
    plot_training=False
)

# Second run: loads existing model (fast!)
print("\n>>> Second call (will load from cache):")
esn2 = auto_load_or_create(
    data=data,
    dt=0.01,
    N_units=20,
    config_dir="./saved_configs",
    auto_save=True,
    plot_training=False
)




In [ ]:

# Verify they're the same
print(f"\n✓ Both models identical:  {np.allclose(esn1.Wout, esn2.Wout)}")
print(f"\n✓ Both models identical:  {np.allclose(esn1.W.toarray(), esn2.W.toarray())}")
print(f"\n✓ Both models identical:  {np.allclose(esn1.Win.toarray(), esn2.Win.toarray())}")
print(f"\n✓ Both models identical:  {np.allclose(esn1.seed, esn2.seed)}")


In [ ]:
esn1.seed

In [ ]:

psi1, _ = esn1.time_integrate(Nt=100)  # Just to show no errors occur
psi2, _ = esn2.time_integrate(Nt=100)  # Just to show no errors occur

In [ ]:
psi1.shape, psi2.shape

In [ ]:

print(f"Predictions identical: {np.allclose(psi1, psi2)}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.plot(psi1[:, 0, 0], label='esn1')
plt.plot(psi2[:, 0, 0], '--', label='esn2')
plt.legend()
plt.title('Predictions from loaded vs original model')
plt.show()

In [ ]:


# ========== Method 2: Manual config management ==========
print("\n" + "=" * 60)
print("Method 2: Manual config management")
print("=" * 60)

# Check if config exists
init_params = {
    'data': data,
    'dt': 0.01,
    'N_units':  50,
    'rho': 0.6,
    'sigma_in': 1.5,
}


query_config = ESNConfig.from_init_params(**init_params)
query_hash = query_config.to_hash()


matching_path = find_matching_config("./saved_configs", "esn_" + query_hash)

if matching_path:
    print(f"\n>>> Found existing config at: {matching_path}")
    config = ESNConfig.load_complete(matching_path)
    esn3 = config.to_esn_model(data=data)
    print("✓ Model loaded successfully")
else:
    print("\n>>> No matching config, training new model...")
    
    esn3 = ESN_model(data=data, dt=0.01, N_units=50, rho=0.6, 
                     sigma_in=1.5, plot_training=False)
    save_esn_model_to_config(esn3, save_dir="./saved_configs", name="esn_" + query_hash)



In [ ]:

# ========== Method 4: List all saved configs ==========
print("\n" + "=" * 60)
print("Method 4: List all saved configs")
print("=" * 60)

from config.esn_config import list_saved_configs

configs = list_saved_configs("./saved_configs", verbose=True)
print(f"\n✓ Total configs found: {len(configs)}")



In [ ]:
configs